In [1]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
target = pd.read_csv("target.csv")

In [2]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [3]:
row = states.shape[0]

print(states.shape)
print(target)

torch.Size([707542, 30, 8, 8])
        value  policy
0          -1     307
1           1    4488
2          -1     657
3           1    4395
4          -1     195
...       ...     ...
707537      1    2538
707538     -1    4488
707539      1    2847
707540     -1    2604
707541      1    3480

[707542 rows x 2 columns]


In [4]:
import sys
sys.path.append('..')

In [5]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [6]:

from core import factory

network = factory.build_network("chess")

In [7]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=1e-3, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [10]:
from core.network import PolicyValueNetwork
from torch import optim
import time
from core.network import PolicyValueNetwork
from torch import optim
import time
import torch
import numpy as np
from torch.optim import lr_scheduler


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          states: torch.Tensor,
          policy: torch.Tensor,
          value: torch.Tensor,
          policy_loss_fn,
          value_loss_fn,
          batch_size: int = 256,
          num_iter: int | None = None,
          duration_hour: float | None = None,
          seed: int = 42):

    if num_iter is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_iter or duration_hour")

    start = time.time()
    rng = np.random.default_rng(seed=seed)
    step = 0

    # Make a LR Scheduler
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=100)

    policy = policy.to(device=DEVICE)
    value = value.unsqueeze(-1).to(device=DEVICE)

    while True:
        if num_iter is not None and step >= num_iter:
            break
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        batch_idx = rng.choice(len(states), batch_size, replace=False)
        batch_states = states[batch_idx]
        batch_policy = policy[batch_idx]
        batch_value  = value[batch_idx]

        optimizer.zero_grad()

        policy_head, value_head = network(batch_states)

        policy_loss = policy_loss_fn(policy_head, batch_policy)
        value_loss  = value_loss_fn(value_head, batch_value)

        loss = policy_loss + value_loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        if step % 10 == 0:
            elapsed = time.time() - start
            print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={value_loss.item():.4f} | {elapsed:.0f}s")

        step += 1

In [ ]:
train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    states=states, 
    policy=policy,
    value=value,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=128,
    seed=42
)

[0] loss=8.0045 | policy=6.4482 | value=1.5563 | 0s
[10] loss=10.0166 | policy=8.3723 | value=1.6443 | 14s
[20] loss=8.3748 | policy=7.7124 | value=0.6624 | 27s
[30] loss=7.9280 | policy=7.4393 | value=0.4887 | 41s
[40] loss=7.9588 | policy=7.3592 | value=0.5996 | 54s
[50] loss=7.5639 | policy=7.0623 | value=0.5016 | 71s
[60] loss=7.5417 | policy=6.9955 | value=0.5461 | 87s
